# sqrt-eps-stabilize — ex1: rescue a BatchNorm-style normalize from divide-by-zero

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `sqrt-eps-stabilize`. Running the final beacon cell reports progress against the `Numerical: sqrt-eps stabilization` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numerical: sqrt-eps stabilization` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sqrt-eps-stabilize`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sqrt-eps-stabilize"
DD_SUBTOPIC = "Numerical: sqrt-eps stabilization"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `sqrt(x + eps)` stabilization — quick refresher

Every BatchNorm / LayerNorm / RMSNorm / Adam / `pairwise_distance` implementation contains the same one-character defense:

```python
normed = x / torch.sqrt(var + eps)        # NOT torch.sqrt(var) — would divide by ~0
```

Without `+ eps`, three things can go wrong when `var` (or any other non-negative quantity) collapses toward zero:
1. `sqrt(0) == 0` → divide-by-zero → `inf` propagates everywhere.
2. `sqrt(tiny_positive)` → numerically unstable gradient — `d/dx sqrt(x) = 1 / (2*sqrt(x))` blows up near 0.
3. `sqrt(tiny_negative_from_float_roundoff)` → `nan` (PyTorch returns NaN on negative inputs, not an exception).

**Where to put the eps.** *Inside* the sqrt (`sqrt(var + eps)`), not after (`sqrt(var) + eps`). The inside-sqrt form keeps the derivative bounded; the outside form does not — it only protects against the divide-by-zero, not against the gradient explosion.

**Typical eps.** `1e-5` for BatchNorm, `1e-6` for LayerNorm/RMSNorm, `1e-8` for Adam's denominator. The values come from empirical stability on float32; pick smaller eps only if you've verified the input range.

### Exercise 1 — rescue a BatchNorm-style normalize from divide-by-zero

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Add the `+ eps` inside-sqrt to a naive normalize so it produces finite output even when the per-channel variance collapses to 0.
> Keywords: batchnorm, stability, eps, nan, inf
> ```

**KCs targeted:** `sqrt-eps-inside-not-outside`, `naive-normalize-produces-nan`

Implement BOTH functions:

1. `ex1_naive_normalize(x)` — the BROKEN version, no eps:
   ```
   mean = x.mean(dim=0)
   var  = x.var(dim=0, unbiased=False)
   return (x - mean) / t.sqrt(var)
   ```
   This will divide by zero whenever any channel has identical values across the batch — common in dead-ReLU channels.

2. `ex1_stable_normalize(x, eps=1e-5)` — the FIXED version. Identical structure, but with `+ eps` **inside the sqrt**:
   `return (x - mean) / t.sqrt(var + eps)`.

`x` has shape `(B, C)` — batch × channels. Both functions return `(B, C)` tensors. The naive version will produce `nan` or `inf` on the test input that has a constant column; the stable version must produce finite values for both inputs.

**Don't try to mask the nans after the fact** — the whole point is to fix the upstream sqrt by putting eps INSIDE it. `t.sqrt(var) + eps` is the wrong fix (only patches the divide-by-zero, not the gradient explosion at small var).

In [ ]:
def ex1_naive_normalize(x: Tensor) -> Tensor:
    mean = x.mean(dim=0)
    var = x.var(dim=0, unbiased=False)
    return (x - mean) / t.sqrt(var)

def ex1_stable_normalize(x: Tensor, eps: float = 1e-5) -> Tensor:
    mean = x.mean(dim=0)
    var = x.var(dim=0, unbiased=False)
    return (x - mean) / t.sqrt(var + eps)


<details><summary>Solution</summary>

```python
def ex1_naive_normalize(x: Tensor) -> Tensor:
    mean = x.mean(dim=0)
    var = x.var(dim=0, unbiased=False)
    return (x - mean) / t.sqrt(var)

def ex1_stable_normalize(x: Tensor, eps: float = 1e-5) -> Tensor:
    mean = x.mean(dim=0)
    var = x.var(dim=0, unbiased=False)
    return (x - mean) / t.sqrt(var + eps)
```

**Why the gradient form matters too.** `d/dx sqrt(x) = 1 / (2 * sqrt(x))`. As `x → 0`, the gradient diverges. With `sqrt(var + eps)` the derivative is bounded by `1 / (2 * sqrt(eps))` — a large number, but finite. With `sqrt(var) + eps`, the gradient is `1 / (2 * sqrt(var))` — still divergent. Only the inside-sqrt form stabilizes BOTH the forward divide AND the backward gradient.

**Why `var(unbiased=False)`.** BatchNorm uses the biased estimator (`/N` not `/(N-1)`) because the per-batch statistics are not samples of a population — they ARE the data we're normalizing. The unbiased correction is for inferring a population variance from a sample, which is a different statistical question.

**What eps to pick.** PyTorch's `nn.BatchNorm2d` defaults to `1e-5`. `nn.LayerNorm` defaults to `1e-5`. RMSNorm typically uses `1e-6`. Adam's denominator uses `1e-8`. The order of magnitude matters more than the exact value — `1e-3` would audibly shift small-variance channels, `1e-12` is rounded to 0 in float32.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()